# SetMe: train the ball detector on a free Colab GPU

**Before you start**
1. Upload `setme-yolo.zip` (made by `setme_vision.labels export`, then zipped) to Google Drive. It looks in *My Drive/Volleyball* first, then anywhere in *My Drive*. A zip or an
   unzipped folder both work.
2. Here in Colab: **Runtime → Change runtime type → T4 GPU → Save**.
3. **Runtime → Run all**. Allow the Google Drive access it asks for.

It takes about 1 to 1.5 hours. Keep this tab open: Colab stops idle sessions. Progress is saved to Drive after
every pass, so if it stops, run all again and it carries on from where it was.

When it's done, download `My Drive/Volleyball/setme-ball/ball.pt` and put it next to your videos.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Find the dataset on Drive (the zip, or a folder if it was unzipped) and copy it to Colab's own disk:
# reading 2,800 images from Drive during training is slow.
!pip -q install ultralytics
import shutil, subprocess
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
VB = next((p for p in DRIVE.iterdir() if p.is_dir() and p.name.lower() == 'volleyball'), DRIVE / 'Volleyball')
OUT = VB / 'setme-ball'                            # where progress and ball.pt are saved
import zipfile
def look(d, any_zip=False):
    zips = [p for p in d.rglob('*') if p.is_file() and ('setme-yolo' in str(p) or (any_zip and p.suffix == '.zip')) and zipfile.is_zipfile(p)]
    yamls = [y for y in d.rglob('data.yaml') if 'setme-yolo' in str(y) or y.parent.name == 'yolo']
    return zips, yamls
zips, yamls = look(VB, any_zip=True) if VB.exists() else ([], [])
if not zips and not yamls:
    zips, yamls = look(DRIVE)                        # not in Volleyball: try the whole of My Drive
shutil.rmtree('/content/yolo', ignore_errors=True)
if zips:
    print('Found', zips[0])
    shutil.copy2(zips[0], '/content/setme-yolo.zip')
    subprocess.run(['unzip', '-q', '-o', '/content/setme-yolo.zip', '-d', '/content/unpacked'], check=True)
    src_dir = next(Path('/content/unpacked').rglob('data.yaml')).parent
elif yamls:
    print('Found the unzipped folder', yamls[0].parent, '(copying 2,800 files from Drive takes a few minutes)')
    src_dir = yamls[0].parent
else:
    print('My Drive has:', sorted(p.name for p in DRIVE.iterdir()))
    for d in [p for p in DRIVE.iterdir() if p.is_dir() and 'volley' in p.name.lower()]:
        for q in sorted(d.rglob('*'))[:40]:
            print('  ', q.relative_to(DRIVE), f'{q.stat().st_size / 1e6:.0f} MB' if q.is_file() else '(folder)')
    raise SystemExit("Couldn't find the dataset. Copy the list above to Claude.")
shutil.copytree(src_dir, '/content/yolo', dirs_exist_ok=True)
Path('/content/yolo/data.yaml').write_text('path: /content/yolo\ntrain: images/train\nval: images/val\nnames:\n  0: ball\n')
print(len(list(Path('/content/yolo/images/train').glob('*.jpg'))), 'training frames,',
      len(list(Path('/content/yolo/images/val').glob('*.jpg'))), 'held-out frames')

In [ ]:
# Train. The settings match setme_vision/train.py: 1280-pixel images because the ball is only ~20 px across,
# and the ball's colour is kept as it is (it's a strong clue).
from ultralytics import YOLO
runs = OUT / 'runs'
last = runs / 'ball' / 'weights' / 'last.pt'
if last.exists():
    print('Carrying on from the last saved pass')
    YOLO(str(last)).train(resume=True)
else:
    YOLO('yolo11n.pt').train(data='/content/yolo/data.yaml', epochs=60, imgsz=1280, batch=16, device=0,
                             project=str(runs), name='ball', exist_ok=True, patience=15, plots=False, workers=2,
                             fliplr=0.5, mosaic=1.0, scale=0.3, degrees=0, hsv_h=0.005)

In [ ]:
# Check it on the two games it never saw, and save ball.pt to Drive.
import shutil
best = runs / 'ball' / 'weights' / 'best.pt'
shutil.copy2(best, OUT / 'ball.pt')
m = YOLO(str(best)).val(data='/content/yolo/data.yaml', imgsz=1280, batch=16, device=0, plots=False)
print(f'On the held-out games: precision {m.box.mp:.2f} (how often a box is really the ball), '
      f'recall {m.box.mr:.2f} (how many balls it finds), mAP50 {m.box.map50:.2f}')
print('Saved: My Drive/Volleyball/setme-ball/ball.pt')